# Chapter 47: Metadata Filtering — Combining Semantic Search with Filters

In Chapter 46, you learned to retrieve the nearest documents using ChromaDB. Now we'll make retrieval more precise by combining semantic similarity with structured metadata filters.
Imagine you're investigating a suspicious payment. You want documents relevant to the query, but only from the AML category. Instead of searching every document, you can filter the candidates by category and then rank the eligible documents by semantic similarity.

# Step 1 — Install dependencies

In [1]:
%pip install chromadb sentence-transformers numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


# Step 2 — Import libraries

In [2]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Step 3 — Create a dataset with metadata

In [3]:
documents = [
    "Banks monitor cross-border payments for suspicious activity.",
    "Money laundering conceals the illegal origin of funds.",
    "Customer due diligence verifies customer identities.",
    "Credit risk models estimate the probability of borrower default.",
    "Machine learning detects unusual financial transaction patterns.",
    "Fraud detection systems identify potentially fraudulent payments.",
    "Banks investigate unusual international wire transfers.",
    "Neural networks classify images and process language."
]

ids = [f"doc_{i}" for i in range(1, 9)]

metadatas = [
    {"category": "AML", "risk_level": "high", "country": "India"},
    {"category": "AML", "risk_level": "high", "country": "UK"},
    {"category": "KYC", "risk_level": "medium", "country": "India"},
    {"category": "Credit Risk", "risk_level": "medium", "country": "US"},
    {"category": "AML", "risk_level": "high", "country": "US"},
    {"category": "Fraud", "risk_level": "high", "country": "India"},
    {"category": "AML", "risk_level": "medium", "country": "India"},
    {"category": "AI", "risk_level": "low", "country": "UK"}
]

print("Documents:", len(documents))
print("Metadata records:", len(metadatas))

Documents: 8
Metadata records: 8


# Step 4 — Generate embeddings

In [4]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = model.encode(
    documents
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (8, 384)


# Step 5 — Create a ChromaDB collection

In [5]:
client = chromadb.EphemeralClient()

collection = client.create_collection(
    name="chapter_47_metadata_filtering",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)

print("Stored records:", collection.count())

Stored records: 8


# Step 6 — Create a query embedding

In [6]:
query = "How can banks identify suspicious international payments?"

query_embedding = model.encode(
    [query]
)

# Step 7 — Run semantic search without filtering

In [7]:
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2)

In [8]:
results

{'ids': [['doc_1', 'doc_7']],
 'embeddings': None,
 'documents': [['Banks monitor cross-border payments for suspicious activity.',
   'Banks investigate unusual international wire transfers.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'AML', 'country': 'India', 'risk_level': 'high'},
   {'category': 'AML', 'country': 'India', 'risk_level': 'medium'}]],
 'distances': [[0.20045453310012817, 0.22564059495925903]]}

In [14]:
for i, documents in enumerate(results):
    print(f"\n Rank: {i+1}")
    print("Document", documents)
    print("metadatas",results["metadatas"])


 Rank: 1
Document ids
metadatas [[{'category': 'AML', 'country': 'India', 'risk_level': 'high'}, {'category': 'AML', 'country': 'India', 'risk_level': 'medium'}]]

 Rank: 2
Document embeddings
metadatas [[{'category': 'AML', 'country': 'India', 'risk_level': 'high'}, {'category': 'AML', 'country': 'India', 'risk_level': 'medium'}]]

 Rank: 3
Document documents
metadatas [[{'category': 'AML', 'country': 'India', 'risk_level': 'high'}, {'category': 'AML', 'country': 'India', 'risk_level': 'medium'}]]

 Rank: 4
Document uris
metadatas [[{'category': 'AML', 'country': 'India', 'risk_level': 'high'}, {'category': 'AML', 'country': 'India', 'risk_level': 'medium'}]]

 Rank: 5
Document included
metadatas [[{'category': 'AML', 'country': 'India', 'risk_level': 'high'}, {'category': 'AML', 'country': 'India', 'risk_level': 'medium'}]]

 Rank: 6
Document data
metadatas [[{'category': 'AML', 'country': 'India', 'risk_level': 'high'}, {'category': 'AML', 'country': 'India', 'risk_level': 'medium'

# Step 8 — Filter by one metadata field

In [15]:
aml_results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    where={"category": "AML"},
    include=["documents", "metadatas", "distances"]
)

for i, document in enumerate(aml_results["documents"][0]):
    print(f"\nRank: {i + 1}")
    print("Document:", document)
    print("Metadata:", aml_results["metadatas"][0][i])
    print("Distance:", round(aml_results["distances"][0][i], 4))


Rank: 1
Document: Banks monitor cross-border payments for suspicious activity.
Metadata: {'risk_level': 'high', 'category': 'AML', 'country': 'India'}
Distance: 0.2005

Rank: 2
Document: Banks investigate unusual international wire transfers.
Metadata: {'country': 'India', 'risk_level': 'medium', 'category': 'AML'}
Distance: 0.2256

Rank: 3
Document: Money laundering conceals the illegal origin of funds.
Metadata: {'country': 'UK', 'category': 'AML', 'risk_level': 'high'}
Distance: 0.4946


# Step 9 — Combine multiple metadata conditions

In [16]:
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5,
    where={
        "$and": [
            {"category": "AML"},
            {"country": "India"}
        ]
    },
    include=["documents", "metadatas", "distances"]
)

for document, metadata in zip(
    results["documents"][0],
    results["metadatas"][0]
):
    print(metadata["country"], "|", metadata["category"])
    print(document, "\n")

India | AML
Banks monitor cross-border payments for suspicious activity. 

India | AML
Banks investigate unusual international wire transfers. 



# Step 10 — Build a reusable filtered-search function

In [27]:
def filtered_search(query, top_k = 3, category = None, country = None):
    if not isinstance(query,str):
        raise ValueError("Query must be a non empty string")

    if top_k < 1:
        raise ValueError("top_k must be atleast 1")

    conditions = []

    if category is not None:
        conditions.append({"category": category})

    if country is not None:
        conditions.append({"country": country})

    query_vector = model.encode([query],normalize_embeddings=True)
    
    search_args = {
        "query_embeddings": query_vector.tolist(),
        "n_results": min(top_k, collection.count()),
        "include": ["documents", "metadatas", "distances"]
    }

    results = collection.query(**search_args)

    output = []


    for i, document in enumerate(results["documents"][0]):
        output.append({
            "document": document,
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return output



In [28]:
filtered_search(
    "Find suspicious international transfers",
    top_k=3,
    category="AML",
    country="India"
)

[{'document': 'Banks investigate unusual international wire transfers.',
  'metadata': {'category': 'AML', 'risk_level': 'medium', 'country': 'India'},
  'distance': 0.2558349370956421},
 {'document': 'Banks monitor cross-border payments for suspicious activity.',
  'metadata': {'category': 'AML', 'country': 'India', 'risk_level': 'high'},
  'distance': 0.3282007575035095},
 {'document': 'Fraud detection systems identify potentially fraudulent payments.',
  'metadata': {'category': 'Fraud', 'country': 'India', 'risk_level': 'high'},
  'distance': 0.5206543207168579}]